In [3]:
import math, random, os
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)

PyTorch: 2.9.0+cpu | Device: cpu


# 8) Transformer — Ngôn ngữ ký tự (Tiny LM)

In [4]:

text=("deep learning changes the way we build intelligent systems. "
      "transformers are powerful for sequence modeling. "
      "rnn and cnn are also useful for many tasks. "
      "this is a tiny corpus for a tiny demo.")
chars=sorted(list(set(text))); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for ch,i in stoi.items()}
vocab_size=len(chars)
def enc(s): return [stoi[c] for c in s]
def dec(ids): return "".join(itos[i] for i in ids)
data=torch.tensor(enc(text), dtype=torch.long)
block=32
def get_batch(B=64):
    ix=torch.randint(len(data)-block-1, (B,))
    x=torch.stack([data[i:i+block] for i in ix])
    y=torch.stack([data[i+1:i+block+1] for i in ix])
    return x.to(device), y.to(device)

class TinyTF(nn.Module):
    def __init__(self, d=64, heads=4, layers=2):
        super().__init__()
        self.tok=nn.Embedding(vocab_size,d)
        self.pos=nn.Embedding(block,d)
        enc_layer=nn.TransformerEncoderLayer(d_model=d, nhead=heads, batch_first=True)
        self.enc=nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.lm=nn.Linear(d, vocab_size)
    def forward(self, idx):
        B,T=idx.size()
        x=self.tok(idx)+self.pos(torch.arange(T, device=idx.device)).unsqueeze(0).expand(B,T,-1)
        x=self.enc(x)
        return self.lm(x)

model=TinyTF().to(device); opt=optim.Adam(model.parameters(), lr=3e-3); lossf=nn.CrossEntropyLoss()
for step in range(800):
    x,y=get_batch(64)
    logits=model(x); loss=lossf(logits.view(-1, vocab_size), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1)%200==0: print(f"step {step+1}: loss={loss.item():.3f}")

@torch.no_grad()
def generate(start="trans", steps=150):
    idx=torch.tensor([enc(start)], dtype=torch.long).to(device)
    for _ in range(steps):
        if idx.size(1)>block: idx=idx[:,-block:]
        logits=model(idx)[:,-1,:]
        probs=torch.softmax(logits,dim=-1)
        nxt=torch.multinomial(probs,1)
        idx=torch.cat([idx,nxt], dim=1)
    return dec(idx[0].tolist())

print("\nSinh văn bản:")
print(generate("trans", 150))

step 200: loss=0.016
step 400: loss=0.006
step 600: loss=0.004
step 800: loss=0.004

Sinh văn bản:
 dmisisis d. tis is is ali isitis
